# 04 — LLM-Assisted MOF Matching (Tier 2)

**Purpose:** For NIST MOFs that could not be matched to CSD entries via DOI (notebook 03),
use a two-step LLM pipeline to identify the correct CSD crystal structure.

**Key design principle:**
The NIST MOF name is not always the name of the MOF being studied in the paper — it may
be a nickname, abbreviation, or lab label. The paper the DOI links to is the ground truth:
it describes the compound in chemical detail. The DOI is our bridge to that information.

**Pipeline:**
1. **DOI → Paper metadata** — fetch title and abstract via OpenAlex for each unmatched MOF.
2. **LLM Step 1 — Formula inference** — given the MOF name and paper context, infer the
   empirical framework formula. The paper abstract is the primary evidence.
3. **Stoichiometric pre-filter** — narrow 132,000 CSD entries to a shortlist using element
   overlap and stoichiometric similarity. Pre-computed at index-build time for speed.
4. **LLM Step 2 — Formula matching** — given the inferred formula, paper context, and CSD
   shortlist, identify the best matching CSD entry with solvent tolerance.

**Outputs:**
- `data/llm_matched_mofs.csv` — confident matches (≥ threshold) ready to merge with notebook 03
- `data/llm_unmatched.csv` — MOFs the pipeline could not resolve (no formula, no candidates,
  low confidence, or pipeline error)

## 1. Setup & Config

In [13]:
import os
import re
import json
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from anthropic import Anthropic
from collections import Counter

load_dotenv(r"data\.env")
assert os.getenv("ANTHROPIC_API_KEY"), "API key not found — check your .env file"

# ── Config ────────────────────────────────────────────────────────────────────
MODEL= "claude-haiku-4-5-20251001"

# Matches below this confidence go to review rather than being accepted outright.
CONFIDENCE_THRESHOLD = 0.5

# Keeping this small reduces cost and keeps the prompt focused.
MAX_CSD_CANDIDATES   = 50

# Minimum fraction of inferred elements that must appear in a CSD formula for that entry to be considered a candidate.
MIN_ELEMENT_OVERLAP  = 0.99

# Polite delay between API calls to avoid rate limiting.
API_DELAY_SECONDS    = 0.3

# Truncate abstract
ABSTRACT_MAX_CHARS   = 2000

client = Anthropic()
print("Anthropic client initialised.")

Anthropic client initialised.


## 2. Load Data

In [2]:
unmatched_df = pd.read_csv('data/unmatched_mofs.csv', encoding='utf-8')

with open('data/mof_adsorbents_for_cif_matching.json', 'r', encoding='utf-8') as f:
    nist_all = json.load(f)
nist_lookup = {m['name']: m for m in nist_all}

csd = pd.read_csv('data/step-02.csv', encoding='utf-8')
csd = csd[csd['note'] == '-'].copy().reset_index(drop=True)

print(f"Unmatched NIST MOFs  : {len(unmatched_df)}")
print(f"CSD entries to search: {len(csd)}")

Unmatched NIST MOFs  : 222
CSD entries to search: 132886


## 3. DOI → Paper Metadata

For each unmatched MOF, fetch the title and abstract of its source paper from
OpenAlex — an open scholarly database with strong chemistry coverage. Abstracts
are stored as an inverted index in OpenAlex and reconstructed into plain text.

Results are cached in memory so that multiple NIST MOFs sharing the same DOI
(multiple isotherms from one paper) only trigger a single network request.

In [3]:
# In-memory cache: doi_string → {'title': ..., 'abstract': ...}
_doi_cache = {}

def reconstruct_abstract_from_openalex(inverted_index: dict) -> str | None:
    if not inverted_index:
        return None
    max_pos = max(pos for positions in inverted_index.values() for pos in positions)
    words   = [''] * (max_pos + 1)
    for word, positions in inverted_index.items():
        for pos in positions:
            words[pos] = word
    return ' '.join(words).strip() or None


def fetch_paper_metadata(doi: str) -> dict:
    if not doi or not isinstance(doi, str):
        return {'title': None, 'abstract': None}
    doi_clean = doi.strip()
    if doi_clean in _doi_cache:
        return _doi_cache[doi_clean]
    result = {'title': None, 'abstract': None}
    try:
        oa_url      = f"https://api.openalex.org/works/doi:{doi_clean}"
        oa_params   = {"mailto": "random@random.com"} 
        oa_response = requests.get(oa_url, params=oa_params, timeout=10)

        if oa_response.status_code == 200:
            oa_data          = oa_response.json()
            result['title']  = oa_data.get('title')
            inverted_index   = oa_data.get('abstract_inverted_index')
            result['abstract'] = reconstruct_abstract_from_openalex(inverted_index)
    except Exception as e:
        print(f"  [OpenAlex warning] {doi_clean}: {e}")
    _doi_cache[doi_clean] = result
    return result

# Quick test on a known MOF paper DOI to confirm network access
test_meta = fetch_paper_metadata("10.1039/B927499e")
print(f"Test fetch — title   : {test_meta['title']}")
if test_meta['abstract']:
    print(f"Test fetch — abstract: {test_meta['abstract'][:100]}...")
else:
    print("Test fetch — abstract: None")

Test fetch — title   : Separation of gas mixtures using Co(ii) carborane-based porous coordination polymers
Test fetch — abstract: Separations of CO(2)/CH(4), CO(2)/N(2), and O(2)/N(2) mixtures were studied in three porous coordina...


## 4. Build CSD Element + Stoichiometry Index

Parse every CSD formula into its element set and element counts at index-build time.
Pre-computing counts here means the stoichiometric ranking step in the pre-filter
does a simple dict lookup per candidate rather than re-parsing 50,000+ formula strings
on every query — a meaningful speedup given the 132,000-entry dataset.

Both `parse_elements` (for the element-set filter) and `parse_element_counts`
(for stoichiometric ranking) live here so this cell is fully self-contained.

In [4]:
def parse_elements(formula: str) -> set:
    if not isinstance(formula, str):
        return set()
    return set(re.findall(r'[A-Z][a-z]?', formula))

def parse_element_counts(formula: str) -> Counter:
    if not isinstance(formula, str):
        return Counter()
    framework_part = formula.split(',')[0]
    framework_part = re.sub(r'(?<![A-Z])n(?!\w)', ' ', framework_part)
    framework_part = re.sub(r'[()]', ' ', framework_part)
    counts = Counter()
    for element, count_str in re.findall(r'([A-Z][a-z]?)(\d*)', framework_part):
        if not element or not element.strip():
            continue
        count = int(count_str) if count_str else 1
        counts[element] += count
    return counts


print("Building CSD element + stoichiometry index...")
csd_index = []
for _, row in csd.iterrows():
    formula = row['formula']
    csd_index.append({
        'identifier'  : row['identifier'],
        'formula'     : formula,
        'has_disorder': row['has_disorder'],
        'elements'    : parse_elements(formula),
        'counts'      : parse_element_counts(formula)   
 })

sample = csd_index[4]
print(f"\nSample entry : {sample['identifier']}")
print(f"  Formula    : {sample['formula']}")
print(f"  Elements   : {sample['elements']}")
print(f"  Counts     : {dict(sample['counts'])}")
print(f"\nIndex built  : {len(csd_index)} entries.")

Building CSD element + stoichiometry index...

Sample entry : ABAGAO
  Formula    : (C14 H13 Cu1 N3 O4)n,H2 O1
  Elements   : {'N', 'C', 'O', 'H', 'Cu'}
  Counts     : {'C': 14, 'H': 13, 'Cu': 1, 'N': 3, 'O': 4}

Index built  : 132886 entries.


## 5. JSON Parser Utility

The LLM occasionally returns malformed JSON — most often because the `reasoning`
field contains apostrophes or parenthetical chemistry notation (e.g. ligand
abbreviations like `H6L(iBu)`) that break standard `json.loads`.

`safe_parse_llm_json` attempts a clean parse first, then falls back to regex
extraction of the structured fields, then returns a safe null result. This
ensures a single bad response never crashes the full pipeline loop.

In [5]:
def safe_parse_llm_json(raw: str) -> dict:
    # Strip markdown fences if present 
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    # Attempt 1: clean parse — works for well-formed responses
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    # Attempt 2: unexpceted syntaxes
    result = {}
    
    # Extract formula (used in infer_formula responses)
    formula_match = re.search(r'"formula"\s*:\s*"([^"]*)"', raw)
    if formula_match:
        val = formula_match.group(1).strip()
        result['formula'] = None if val.lower() == 'null' else val
    
    # Extract best_match (used in match_formula responses)
    match_match = re.search(r'"best_match"\s*:\s*"([^"]*)"', raw)
    if match_match:
        val = match_match.group(1).strip()
        result['best_match'] = None if val.lower() == 'null' else val
    
    # Extract confidence — this is a number so it's never malformed
    conf_match = re.search(r'"confidence"\s*:\s*([0-9.]+)', raw)
    if conf_match:
        result['confidence'] = float(conf_match.group(1))
    
    # For reasoning, extract what we can but don't crash if it's malformed —
    # truncate at the first problematic character if needed
    reasoning_match = re.search(r'"reasoning"\s*:\s*"(.*?)(?:"|$)', raw, re.DOTALL)
    result['reasoning'] = reasoning_match.group(1)[:200] if reasoning_match else 'reasoning unparseable'
    
    if 'confidence' in result:
        return result
    
    # Attempt 3: total failure — return a safe null response so the loop
    # continues rather than crashing. The MOF will be routed to unmatched.
    return {'formula': None, 'best_match': None, 'confidence': 0.0, 
            'reasoning': f'JSON parse failed entirely: {raw[:100]}'}

## 6. LLM Step 1 — Formula Inference

Given the NIST MOF name and the paper title/abstract, the LLM infers the most
likely empirical framework formula. The paper context is the primary signal —
the MOF name is secondary, since it may not match what the paper actually studies.

The formula is returned in CSD-style spacing (`C18 H6 Cu3 O12`) so it can be
parsed directly by the element and stoichiometry functions in section 4.

In [6]:
FORMULA_INFERENCE_SYSTEM = """You are an expert in metal-organic framework (MOF) chemistry.

You will be given a MOF name from the NIST Isotherm Database, along with the title
and abstract of the scientific paper that the NIST entry links to via its DOI.

IMPORTANT: The NIST MOF name is not always the name of the MOF being studied in the
paper. It may be a nickname, an abbreviation, or a label that differs from how the
compound is described in the paper. The paper title and abstract are your most reliable
source of chemical identity — use them as your primary evidence, and treat the MOF name
as a secondary clue.

Your task: deduce the most likely empirical molecular formula of the MOF framework
described in the paper (excluding any guest or solvent molecules).

Rules:
- Return the formula using CSD-style notation with spaces between element-count pairs,
  e.g. 'C18 H6 Cu3 O12' or 'C8 H4 O5 Zn1'
- Include only the framework atoms — exclude solvents (DMF, H2O, EtOH, MeCN, etc.)
- Draw on the metal and linker information in the abstract to determine the formula
- If the abstract describes multiple MOFs, infer the formula of the primary compound
- If you cannot make a reasonable inference even with the paper context, return null

Respond ONLY with valid JSON. No preamble, no markdown.
Schema:
{
  "formula": "C18 H6 Cu3 O12 or null",
  "confidence": 0.0 to 1.0,
  "reasoning": "one sentence explaining how you derived the formula"
}"""


def infer_formula(nist_name: str, paper_metadata: dict) -> dict:
    # Build the user message, including paper fields only if they exist
    parts = [f"MOF name: {nist_name}"]

    if paper_metadata.get('title'):
        parts.append(f"Paper title: {paper_metadata['title']}")

    if paper_metadata.get('abstract'):
        abstract = paper_metadata['abstract'][:ABSTRACT_MAX_CHARS]
        parts.append(f"Paper abstract: {abstract}")

    user_message = "\n".join(parts)

    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        system=FORMULA_INFERENCE_SYSTEM,
        messages=[{"role": "user", "content": user_message}]
    )
    raw = response.content[0].text.strip()
    return safe_parse_llm_json(raw)

## 7. Stoichiometric Pre-filter

Narrows the 132,000 CSD entries to a shortlist for the matching LLM using two stages:

**Stage 1 — Element-set filter:** rules out entries missing key elements entirely
(e.g. no copper when the inferred formula contains Cu). Fast set intersection,
no API cost. `MIN_ELEMENT_OVERLAP` is deliberately permissive to avoid false exclusions.

**Stage 2 — Stoichiometric ranking:** scores remaining candidates by how closely
their element *ratios* match the inferred formula after normalisation. Normalisation
handles unit cell multiplicity — `C18 H6 Cu3 O12` and `C36 H12 Cu6 O24` score
identically since they describe the same compound. Candidates are ranked by a
weighted combination of element overlap (0.3) and stoichiometric similarity (0.7),
ensuring the top 15 are spread across the full database rather than sorted alphabetically.

In [7]:
def normalise_counts(counts: Counter) -> dict:
    if not counts:
        return {}
    valid = {el: c for el, c in counts.items() if c > 0}
    if not valid:
        return {}
    min_count = min(valid.values())
    if min_count == 0:
        return {}
    return {el: c / min_count for el, c in valid.items()}


def stoichiometric_similarity(inferred_counts: Counter, csd_counts: Counter) -> float:
    # Scores how closely two formula stoichiometries match after normalisation.
    if not inferred_counts or not csd_counts:
        return 0.0
    inferred_norm = normalise_counts(inferred_counts)
    csd_norm      = normalise_counts(csd_counts)
    if not inferred_norm or not csd_norm:
        return 0.0
    scores = []
    for element, inferred_ratio in inferred_norm.items():
        csd_ratio  = csd_norm.get(element, 0.0)
        difference = abs(inferred_ratio - csd_ratio)
        scores.append(1.0 / (1.0 + difference))
    return sum(scores) / len(scores) if scores else 0.0


def get_formula_candidates(inferred_formula: str) -> list:
    query_elements = parse_elements(inferred_formula)
    query_counts   = parse_element_counts(inferred_formula)
    if not query_elements:
        return []
    # Stage 1: broad element-set filter
    element_filtered = []
    for entry in csd_index:
        if not entry['elements']:
            continue
        overlap = len(query_elements & entry['elements']) / len(query_elements)
        if overlap >= MIN_ELEMENT_OVERLAP:
            element_filtered.append((overlap, entry))
    # Stage 2: stoichiometric ranking
    scored_candidates = []
    for element_overlap, entry in element_filtered:
        stoich_score   = stoichiometric_similarity(query_counts, entry['counts'])
        combined_score = 0.3 * element_overlap + 0.7 * stoich_score
        scored_candidates.append({
            **entry,
            'element_overlap': round(element_overlap, 3),
            'stoich_score'   : round(stoich_score, 3),
            'combined_score' : round(combined_score, 3)
        })
    scored_candidates.sort(key=lambda x: -x['combined_score'])
    return scored_candidates[:MAX_CSD_CANDIDATES]


## 8. LLM Step 2 — Formula Matching with Solvent Tolerance

Given the inferred formula, the pre-filtered CSD shortlist, and the paper context,
the LLM identifies the best structural match. The paper abstract serves as a
chemical tie-breaker when multiple CSD candidates share similar formulas.

Key instructions to the model:
- CSD formulas include co-crystallised solvents after a comma — ignore them
- The `n` suffix denotes a polymeric framework — ignore it
- Small H-count differences are acceptable (protonation states, guest water)
- Paper context takes priority over formula similarity when they conflict

In [8]:
FORMULA_MATCHING_SYSTEM = """You are an expert in metal-organic framework (MOF) crystal chemistry.
You are matching a MOF from the NIST Isotherm Database to a crystal structure entry
in the Cambridge Structural Database (CSD).

You will be given:
- The NIST MOF name and inferred framework formula
- The title and abstract of the paper the NIST entry links to (your primary evidence)
- A shortlist of CSD candidates pre-filtered by element overlap

IMPORTANT: The NIST MOF name may not match the compound described in the paper.
Use the paper title and abstract as your primary chemical evidence when deciding
which CSD entry corresponds to the same compound. The inferred formula is a
supporting clue, not the sole criterion.

CSD formula interpretation rules:
- Content after a comma is co-crystallised solvent (e.g. ',2(H2 O1)' is water of
  crystallisation) — ignore it when comparing to the solvent-free framework formula.
- The 'n' suffix denotes a polymeric/extended framework — ignore it.
- Prefixes like '2(' mean two formula units in the cell — normalise before comparing.
- Small differences in H count are acceptable (protonation states, guest H2O).

If no candidate is a plausible match even after considering the paper context,
return null with a low confidence score.

Respond ONLY with valid JSON. No preamble, no markdown.
Schema:
{
  "best_match": "CSD_IDENTIFIER or null",
  "confidence": 0.0 to 1.0,
  "reasoning": "one or two sentences explaining the match decision"
}"""


def match_formula(nist_name: str, inferred_formula: str,
                  candidates: list, paper_metadata: dict) -> dict:
    # In match_formula, update the candidates_text line to show all three scores
    candidates_text = "\n".join(
        f"  - {c['identifier']}: {c['formula']}  "
        f"(element_overlap={c['element_overlap']:.2f}, "
        f"stoich_score={c['stoich_score']:.2f})"
        for c in candidates
    )
    # Build the user message — paper fields included only when available
    parts = [
        f"NIST MOF name: {nist_name}",
        f"Inferred framework formula: {inferred_formula}"
    ]
    if paper_metadata.get('title'):
        parts.append(f"Source paper title: {paper_metadata['title']}")
    if paper_metadata.get('abstract'):
        abstract = paper_metadata['abstract'][:ABSTRACT_MAX_CHARS]
        parts.append(f"Source paper abstract: {abstract}")
    parts.append(f"\nCSD candidates (pre-filtered by element overlap):\n{candidates_text}")
    parts.append("\nWhich CSD entry best matches the MOF described in the paper? Respond with JSON only.")
    user_message = "\n".join(parts)
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=FORMULA_MATCHING_SYSTEM,
        messages=[{"role": "user", "content": user_message}]
    )
    raw = response.content[0].text.strip()
    return safe_parse_llm_json(raw)
    

## 9. Pipeline Test

Runs the full two-step pipeline on the first 2 unmatched MOFs before committing
to the full loop. Verifies the API connection, DOI fetch, formula inference,
stoichiometric pre-filter, and LLM matching all work on real data.

In [ ]:
# ── Test Across Selected MOFs ───────────────────────────────

for idx in range(2):
    test_name   = unmatched_df.iloc[idx]['name']
    test_record = nist_lookup.get(test_name, {})
    test_doi    = test_record.get('DOI', None)

    print(f"{'='*60}")
    print(f"[{idx}] MOF  : {test_name}")
    print(f"     DOI  : {test_doi}")

    # Fetch paper metadata via OpenAlex/Semantic Scholar/Crossref cascade
    test_paper = fetch_paper_metadata(test_doi)
    print(f"     Title: {test_paper.get('title', 'None')}")
    has_abstract = bool(test_paper.get('abstract'))
    print(f"     Abstract: {'yes — ' + test_paper['abstract'][:120] + '...' if has_abstract else 'none'}")

    # Step 1: infer molecular formula from MOF name + paper context
    print("\n  ── Step 1: Formula inference ──")
    inferred = infer_formula(test_name, test_paper)
    print(f"  Formula   : {inferred.get('formula')}  (conf={inferred.get('confidence', 0):.2f})")
    print(f"  Reasoning : {inferred.get('reasoning', '')[:120]}")
    time.sleep(API_DELAY_SECONDS)

    if not inferred.get('formula'):
        print("  → No formula inferred — would be marked unmatched.\n")
        continue

    # Pre-filter: narrow CSD by element + stoichiometric similarity
    print("\n  ── Pre-filter: CSD candidates ──")
    candidates = get_formula_candidates(inferred['formula'])
    print(f"  Candidates found: {len(candidates)}")
    for c in candidates:
        print(f"    {c['identifier']:12s}  overlap={c['element_overlap']:.2f}  "
              f"stoich={c['stoich_score']:.2f}  {c['formula']}")

    if not candidates:
        print("  → No candidates passed the filter — would be marked unmatched.\n")
        continue

    # Step 2: LLM matches formula + paper context against CSD candidates
    print("\n  ── Step 2: LLM formula matching ──")
    match_result = match_formula(test_name, inferred['formula'], candidates, test_paper)
    print(f"  Best match : {match_result.get('best_match')}")
    print(f"  Confidence : {match_result.get('confidence', 0):.2f}")
    print(f"  Reasoning  : {match_result.get('reasoning', '')[:100]}")
    time.sleep(API_DELAY_SECONDS)

    print()

[0] MOF  : SDU-8
     DOI  : 10.1021/Ic3015207
     Title: Comparison of the Effect of Functional Groups on Gas-Uptake Capacities by Fixing the Volumes of Cages A and B and Modifying the Inner Wall of Cage C in rht-Type MOFs
     Abstract: yes — Three porous (3,24)-connected rht-type metal-organic frameworks (MOFs), [Cu(3)L(H(2)O)(3)]·xsolvents (H(6)L(OH) = 4,4',4...

  ── Step 1: Formula inference ──
  Formula   : C78 H54 Cu3 O18 Si1  (conf=0.85)
  Reasoning : SDU-8 contains Cu3 paddlewheel SBUs paired with C3-symmetric hexacarboxylate ligands with an isobutyl-substituted Si cen

  ── Pre-filter: CSD candidates ──
  Candidates found: 50
    OFALEP        overlap=1.00  stoich=0.44  (C48 H48 Cu3 Mo12 N24 O40 Si1 2-)n,n(C12 H12 Cu1 N6 O1 2+),3(H2 O1)
    HINZOX        overlap=1.00  stoich=0.44  (C40 H40 Cu3 F6 N4 O8 Si1)n
    JISSEN        overlap=1.00  stoich=0.43  (C36 H42 Cu3 F6 N18 O6 Si1 4+)n,F6 Si1 2-,B1 F4 1-,Cl1 1-
    OJAQID        overlap=1.00  stoich=0.44  2(C44 H47 Cu3 Mo12 N

## 10. Full Pipeline Loop

Iterates over all unmatched MOFs. For each one:
1. Fetches paper metadata from OpenAlex via the primary DOI (cached)
2. LLM Step 1 infers the empirical framework formula from name + paper context
3. Stoichiometric pre-filter narrows 132,000 CSD entries to a shortlist
4. LLM Step 2 identifies the best CSD match from the shortlist

All outcomes are recorded explicitly — formula inference failure, no candidates,
successful match, and pipeline errors all produce a result row, so nothing is
silently dropped and the output is fully auditable.

In [16]:
results = []
total   = len(unmatched_df)

for i, row in unmatched_df.iterrows():
    nist_name = row['name']
    hashkey   = row.get('hashkey', None)

    # Retrieve the original NIST record to get the primary DOI
    nist_record = nist_lookup.get(nist_name, {})
    primary_doi = nist_record.get('DOI', None)

    # Fetch paper metadata — cached after first call so no duplicate requests
    paper_metadata = fetch_paper_metadata(primary_doi)
    has_title    = bool(paper_metadata.get('title'))
    has_abstract = bool(paper_metadata.get('abstract'))

    print(
        f"[{i+1}/{total}] {nist_name} | "
        f"title={'yes' if has_title else 'no'}  "
        f"abstract={'yes' if has_abstract else 'no'}"
    )

    try:
        # ── Step 1: infer formula from name + paper context ──────────────────
        inferred = infer_formula(nist_name, paper_metadata)
        time.sleep(API_DELAY_SECONDS)

        inferred_formula     = inferred.get('formula')
        inference_confidence = inferred.get('confidence', 0.0)
        inference_reasoning  = inferred.get('reasoning', '')

        if not inferred_formula:
            # LLM could not infer a formula even with paper context — record and move on
            results.append({
                'nist_name'           : nist_name,
                'nist_hashkey'        : hashkey,
                'doi'                 : primary_doi,
                'paper_title'         : paper_metadata.get('title'),
                'inferred_formula'    : None,
                'csd_identifier'      : None,
                'match_confidence'    : 0.0,
                'inference_confidence': inference_confidence,
                'reasoning'           : f'Formula inference failed: {inference_reasoning}',
                'source'              : 'llm_formula'
            })
            print(f"  No formula inferred")
            continue

        print(f"  Inferred: {inferred_formula} (conf={inference_confidence:.2f})")

        # ── Pre-filter: narrow CSD by element overlap ────────────────────────
        candidates = get_formula_candidates(inferred_formula)

        if not candidates:
            results.append({
                'nist_name'           : nist_name,
                'nist_hashkey'        : hashkey,
                'doi'                 : primary_doi,
                'paper_title'         : paper_metadata.get('title'),
                'inferred_formula'    : inferred_formula,
                'csd_identifier'      : None,
                'match_confidence'    : 0.0,
                'inference_confidence': inference_confidence,
                'reasoning'           : 'No CSD candidates passed element overlap filter',
                'source'              : 'llm_formula'
            })
            print(f"  No element candidates found")
            continue

        # ── Step 2: LLM matches formula + paper context against CSD candidates
        match = match_formula(nist_name, inferred_formula, candidates, paper_metadata)
        time.sleep(API_DELAY_SECONDS)

        results.append({
            'nist_name'           : nist_name,
            'nist_hashkey'        : hashkey,
            'doi'                 : primary_doi,
            'paper_title'         : paper_metadata.get('title'),
            'inferred_formula'    : inferred_formula,
            'csd_identifier'      : match.get('best_match'),
            'match_confidence'    : match.get('confidence', 0.0),
            'inference_confidence': inference_confidence,
            'reasoning'           : (
                f"[Formula] {inference_reasoning} | "
                f"[Match] {match.get('reasoning', '')}"
            ),
            'source'              : 'llm_formula'
        })
        print(
            f"  Match: {match.get('best_match')} "
            f"(conf={match.get('confidence', 0):.2f})"
        )

    except Exception as e:
        results.append({
            'nist_name'           : nist_name,
            'nist_hashkey'        : hashkey,
            'doi'                 : primary_doi,
            'paper_title'         : paper_metadata.get('title'),
            'inferred_formula'    : None,
            'csd_identifier'      : None,
            'match_confidence'    : 0.0,
            'inference_confidence': 0.0,
            'reasoning'           : f'Pipeline error: {str(e)}',
            'source'              : 'llm_formula'
        })
        print(f"  ERROR: {e}")

print(f"\nLoop complete. {len(results)} results recorded.")

[1/222] SDU-8 | title=yes  abstract=yes
  Inferred: C54 H36 Cu3 O12 Si1 (conf=0.85)
  Match: None (conf=0.15)
[2/222] [Co(2)(pyridine)2(H2O)] | title=yes  abstract=yes
  No formula inferred
[3/222] Sc2(O2CC6H4CO2)3 | title=yes  abstract=yes
  Inferred: C18 H6 Sc2 O12 (conf=0.98)
  Match: WEMKAE (conf=0.85)
[4/222] Co14-MOF-74 | title=yes  abstract=no
  Inferred: C12 H6 Co2 O12 (conf=0.75)
  Match: MAQDIS (conf=0.72)
[5/222] opt-UiO-66(Zr)-(OH)2 | title=yes  abstract=yes
  Inferred: C12 H6 Zr3 O12 (conf=0.85)
  Match: BOHKAM (conf=0.75)
[6/222] NOTT-102a | title=yes  abstract=yes
  No formula inferred
[7/222] MIL-53-TDC | title=yes  abstract=no
  Inferred: C8 H4 Al1 O6 (conf=0.60)
  Match: EMAGOR (conf=0.92)
[8/222] CGUC-0.5-6 | title=no  abstract=no
  No formula inferred
[9/222] MOF-1C' | title=yes  abstract=no
  No formula inferred
[10/222] Zn(Im)1.25(5-ClbIM)0.75 | title=yes  abstract=no
  Inferred: C9 H7 Cl0.75 N8 Zn1 (conf=0.72)
  Match: YATWOG (conf=0.72)
[11/222] JUC-569 | title=

## 11. Export Results

In [17]:
df_results = pd.DataFrame(results)

# Attach CSD formula and disorder flag to rows that produced a match
csd_meta = csd[['identifier', 'formula', 'has_disorder']].drop_duplicates(subset='identifier')
csd_meta = csd_meta.rename(columns={'identifier': 'csd_identifier', 'formula': 'csd_formula'})
df_results = df_results.merge(csd_meta, on='csd_identifier', how='left')

# Split into two tiers: confident matches and everything else
has_match    = df_results['csd_identifier'].notna()
df_matched   = df_results[has_match & (df_results['match_confidence'] >= CONFIDENCE_THRESHOLD)]
df_unmatched = df_results[~has_match | (df_results['match_confidence'] < CONFIDENCE_THRESHOLD)]

df_matched.to_csv('data/llm_matched_mofs.csv',   index=False, encoding='utf-8')
df_unmatched.to_csv('data/llm_unmatched.csv',    index=False, encoding='utf-8')

print(f"{'='*55}")
print(f"RESULTS SUMMARY (confidence threshold = {CONFIDENCE_THRESHOLD})")
print(f"{'='*55}")
print(f"Total processed  : {len(df_results):>4}")
print(f"Matched          : {len(df_matched):>4}  -> data/llm_matched_mofs.csv")
print(f"Unmatched        : {len(df_unmatched):>4}  -> data/llm_unmatched.csv")

if len(df_matched) > 0:
    print(f"\nConfidence distribution (matched):")
    print(df_matched['match_confidence'].describe().round(3))

# Breakdown of why MOFs ended up unmatched — useful for diagnosing pipeline gaps
if len(df_unmatched) > 0:
    print(f"\nUnmatched breakdown:")
    no_formula   = df_unmatched['inferred_formula'].isna().sum()
    no_candidate = (df_unmatched['inferred_formula'].notna() &
                    df_unmatched['csd_identifier'].isna()).sum()
    low_conf     = (df_unmatched['csd_identifier'].notna()).sum()
    print(f"  No formula inferred   : {no_formula}")
    print(f"  No CSD candidates     : {no_candidate}")
    print(f"  Low confidence match  : {low_conf}")


RESULTS SUMMARY (confidence threshold = 0.5)
Total processed  :  222
Matched          :  128  -> data/llm_matched_mofs.csv
Unmatched        :   94  -> data/llm_unmatched.csv

Confidence distribution (matched):
count    128.000
mean       0.827
std        0.094
min        0.720
25%        0.720
50%        0.850
75%        0.920
max        0.950
Name: match_confidence, dtype: float64

Unmatched breakdown:
  No formula inferred   : 46
  No CSD candidates     : 46
  Low confidence match  : 2
